In [1]:
!pip install Lightgbm

In [2]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 7.9 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

import lightgbm as lgb
import optuna
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 한글 폰트 설치


In [4]:
[f.name for f in fm.fontManager.ttflist if "Nanum" in f.name]

[]

In [5]:
!apt-get install -y fonts-nanum

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  fonts-nanum
0 upgraded, 1 newly installed, 0 to remove and 3 not upgraded.
Need to get 10.3 MB of archives.
After this operation, 34.1 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 fonts-nanum all 20200506-1 [10.3 MB]
Fetched 10.3 MB in 0s (20.9 MB/s)
Selecting previously unselected package fonts-nanum.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../fonts-nanum_20200506-1_all.deb ...
Unpacking fonts-nanum (20200506-1) ...
Setting up fonts-nanum (20200506-1) ...
Processing triggers for fontconfig (2.13.1-4.2ubuntu5) ...


In [6]:
!fc-cache -fv
!rm -rf ~/.cache/matplotlib

/usr/share/fonts: caching, new cache contents: 0 fonts, 1 dirs
/usr/share/fonts/truetype: caching, new cache contents: 0 fonts, 3 dirs
/usr/share/fonts/truetype/humor-sans: caching, new cache contents: 1 fonts, 0 dirs
/usr/share/fonts/truetype/liberation: caching, new cache contents: 16 fonts, 0 dirs
/usr/share/fonts/truetype/nanum: caching, new cache contents: 12 fonts, 0 dirs
/usr/local/share/fonts: caching, new cache contents: 0 fonts, 0 dirs
/root/.local/share/fonts: skipping, no such directory
/root/.fonts: skipping, no such directory
/usr/share/fonts/truetype: skipping, looped directory detected
/usr/share/fonts/truetype/humor-sans: skipping, looped directory detected
/usr/share/fonts/truetype/liberation: skipping, looped directory detected
/usr/share/fonts/truetype/nanum: skipping, looped directory detected
/var/cache/fontconfig: cleaning cache directory
/root/.cache/fontconfig: not cleaning non-existent cache directory
/root/.fontconfig: not cleaning non-existent cache director

# 데이터 불러오기

In [7]:
# 데이터 불러오기
df = pd.read_csv("final_dataset_line1_8_230101-241231.csv")

print(df.head())
print(df.columns)

FileNotFoundError: [Errno 2] No such file or directory: 'final_dataset_line1_8_230101-241231.csv'

In [ ]:
print(df.isnull().sum())

# 날짜처리 함수

In [ ]:
if "날짜" in df.columns:
    df["날짜"] = pd.to_datetime(df["날짜"])

    df["year"] = df["날짜"].dt.year
    df["month"] = df["날짜"].dt.month
    df["day"] = df["날짜"].dt.day
    df["weekday"] = df["날짜"].dt.weekday

# 원본 날짜 컬럼 제거
df = df.drop(columns=["날짜"])

# 불필요 컬럼 삭제

In [ ]:
# 'month' 컬럼을 삭제하는 코드
df = df.drop('month', axis=1)

# 문자열 컬럼을 카테고리로 변경

In [ ]:
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].astype("category")

# 모델 함수 정의

In [ ]:
# def train_model(df, target_col):

#     # 다른 타겟 제거
#     drop_cols = []

#     if target_col == "승차인원":
#         drop_cols = ["하차인원"]

#     elif target_col == "하차인원":
#         drop_cols = ["승차인원"]

#     # feature / target 생성

#     X = df.drop(columns=drop_cols + [target_col])

#     # datetime 컬럼 제거
#     datetime_cols = X.select_dtypes(
#         include=['datetime64']
#     ).columns

#     X = X.drop(columns=datetime_cols)

#     y = df[target_col]

#     # train / test split

#     X_train, X_test, y_train, y_test = train_test_split(
#         X,
#         y,
#         test_size=0.2,
#         random_state=42
#     )

#     # Optuna 목적 함수

#     def objective(trial):

#         params = {

#             "objective": "regression",
#             "metric": "mae",
#             "boosting_type": "gbdt",

#             "n_estimators": trial.suggest_int(
#                 "n_estimators",
#                 100,
#                 1250
#             ),

#             "learning_rate": trial.suggest_float(
#                 "learning_rate",
#                 0.005,
#                 0.1,
#                 log=True
#             ),

#             "num_leaves": trial.suggest_int(
#                 "num_leaves",
#                 20,
#                 200
#             ),

#             "max_depth": trial.suggest_int(
#                 "max_depth",
#                 3,
#                 15
#             ),

#             "min_child_samples": trial.suggest_int(
#                 "min_child_samples",
#                 5,
#                 100
#             ),

#             "subsample": trial.suggest_float(
#                 "subsample",
#                 0.5,
#                 1.0
#             ),

#             "colsample_bytree": trial.suggest_float(
#                 "colsample_bytree",
#                 0.5,
#                 1.0
#             ),

#             "reg_alpha": trial.suggest_float(
#                 "reg_alpha",
#                 1e-3,
#                 10.0,
#                 log=True
#             ),

#             "reg_lambda": trial.suggest_float(
#                 "reg_lambda",
#                 1e-3,
#                 10.0,
#                 log=True
#             ),

#             "random_state": 42,
#             "verbosity": -1
#         }

#         model = lgb.LGBMRegressor(**params)

#         model.fit(
#             X_train,
#             y_train,

#             eval_set=[(X_test, y_test)],

#             eval_metric="mae",

#             callbacks=[
#                 lgb.early_stopping(100),
#                 lgb.log_evaluation(0)
#             ]
#         )

#         pred = model.predict(X_test)

#         mae = mean_absolute_error(
#             y_test,
#             pred
#         )

#         return mae

#     # Optuna 실행

#     study = optuna.create_study(
#         direction="minimize"
#     )

#     study.optimize(
#         objective,
#         n_trials=30
#     )

#     print("\n최적 파라미터")
#     print(study.best_params)

#     # 최종 모델

#     best_model = lgb.LGBMRegressor(
#         **study.best_params,
#         objective="regression",
#         random_state=42
#     )

#     best_model.fit(
#         X_train,
#         y_train
#     )

#     # 예측

#     pred = best_model.predict(X_test)

#     # 평가

#     mae = mean_absolute_error(
#         y_test,
#         pred
#     )

#     mse = mean_squared_error(
#         y_test,
#         pred
#     )

#     rmse = np.sqrt(mse)

#     print("\n====================")
#     print(f"{target_col} 결과")
#     print("====================")

#     print("MAE :", mae)
#     print("MSE :", mse)
#     print("RMSE :", rmse)


#     # return


#     return (
#         best_model,
#         X_train,
#         X_test,
#         y_train,
#         y_test
#     )

In [ ]:
def train_model(df, target_col):

    import lightgbm as lgb
    import optuna
    import numpy as np

    from sklearn.model_selection import train_test_split
    from sklearn.metrics import mean_absolute_error, mean_squared_error

    # ------------------------
    # 타겟 분리
    # ------------------------
    drop_cols = []

    if target_col == "승차인원":
        drop_cols = ["하차인원"]
    elif target_col == "하차인원":
        drop_cols = ["승차인원"]

    X = df.drop(columns=drop_cols + [target_col])

    # datetime 제거
    datetime_cols = X.select_dtypes(include=['datetime64']).columns
    X = X.drop(columns=datetime_cols)

    y = df[target_col]

    # ------------------------
    # train / test split
    # ------------------------
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )

    # ------------------------
    # 🔥 핵심: best 모델 저장용
    # ------------------------
    best_model_holder = {"model": None}
    best_score_holder = {"score": float("inf")}

    # ------------------------
    # Optuna 목적 함수
    # ------------------------
    def objective(trial):

        params = {
            "objective": "regression",
            "metric": "mae",
            "boosting_type": "gbdt",

            # 🔥 속도 최적화
            "n_estimators": trial.suggest_int("n_estimators", 100, 500),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),

            "num_leaves": trial.suggest_int("num_leaves", 20, 100),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 50),

            "subsample": trial.suggest_float("subsample", 0.7, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),

            "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 1.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 1.0, log=True),

            "random_state": 42,
            "verbosity": -1
        }

        model = lgb.LGBMRegressor(**params)

        model.fit(
            X_train,
            y_train,
            eval_set=[(X_test, y_test)],
            eval_metric="mae",
            callbacks=[
                lgb.early_stopping(50),
                lgb.log_evaluation(0)
            ]
        )

        pred = model.predict(X_test)
        mae = mean_absolute_error(y_test, pred)

        # 🔥 최고 모델 저장
        if mae < best_score_holder["score"]:
            best_score_holder["score"] = mae
            best_model_holder["model"] = model

        return mae

    # ------------------------
    # Optuna 실행 (🔥 속도 줄임)
    # ------------------------
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=10)   # 🔥 원래 30 → 10

    print("\n최적 파라미터")
    print(study.best_params)

    # ------------------------
    # 🔥 재학습 제거
    # ------------------------
    best_model = best_model_holder["model"]

    # ------------------------
    # 예측
    # ------------------------
    pred = best_model.predict(X_test)

    # ------------------------
    # 평가
    # ------------------------
    mae = mean_absolute_error(y_test, pred)
    mse = mean_squared_error(y_test, pred)
    rmse = np.sqrt(mse)

    print("\n====================")
    print(f"{target_col} 결과")
    print("====================")

    print("MAE :", mae)
    print("MSE :", mse)
    print("RMSE :", rmse)

    # ------------------------
    # return
    # ------------------------
    return best_model, X_train, X_test, y_train, y_test

In [ ]:
# 승차 모델
ride_model, X_train_r, X_test_r, y_train_r, y_test_r = train_model(df, "승차인원")

# 하차 모델
off_model, X_train_o, X_test_o, y_train_o, y_test_o = train_model(df, "하차인원")

In [ ]:
# 승차 예측
ride_pred = ride_model.predict(X_test_r)

# 하차 예측
off_pred = off_model.predict(X_test_o)

print(ride_pred[:10])
print(off_pred[:10])

In [ ]:
from sklearn.metrics import mean_absolute_error

# 승차
ride_train_pred = ride_model.predict(X_train_r)
ride_test_pred = ride_model.predict(X_test_r)

print("승차 Train MAE:", mean_absolute_error(y_train_r, ride_train_pred))
print("승차 Test MAE :", mean_absolute_error(y_test_r, ride_test_pred))

# 하차
off_train_pred = off_model.predict(X_train_o)
off_test_pred = off_model.predict(X_test_o)

print("하차 Train MAE:", mean_absolute_error(y_train_o, off_train_pred))
print("하차 Test MAE :", mean_absolute_error(y_test_o, off_test_pred))

In [ ]:
import pickle

# ✅ 승차 모델 저장
ride_data = {
    "model": ride_model,
    "columns": X_train_r.columns
}

with open("/content/ride_model.pkl", "wb") as f:
    pickle.dump(ride_data, f)


# ✅ 하차 모델 저장
off_data = {
    "model": off_model,
    "columns": X_train_o.columns
}

with open("/content/off_model.pkl", "wb") as f:
    pickle.dump(off_data, f)

print("승차 / 하차 모델 각각 저장 완료")

In [ ]:
from google.colab import files

files.download("/content/ride_model.pkl")
files.download("/content/off_model.pkl")